<a href="https://colab.research.google.com/github/rajputaman123/tata-steel-machine-failure-prediction/blob/main/TataSteel_Machine_Failure_Prediction_ML_Submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - Tata Steel Machine Failure Prediction



##### **Project Type**    - Classification
##### **Contribution**    - Individual
##### **Team Member 1 -** Aman Kumar


# **Project Summary -**

TATA Steel, a leader in the steel manufacturing industry, relies on
a large fleet of heavy machinery to maintain production quality and
minimize downtime. Unplanned machine failures lead to significant
production losses, increased maintenance costs, and safety risks.
This project aims to build a predictive maintenance system that
identifies the likelihood of machine failure based on real-time
operational sensor data, enabling proactive maintenance instead of
reactive repairs.

The dataset used is a synthetically generated but realistic dataset
(based on the AI4I 2020 Predictive Maintenance benchmark) containing
136,429 training records with 14 features, including air
temperature, process temperature, rotational speed, torque, tool
wear, and product quality type (Low/Medium/High). The target
variable, "Machine failure," is a binary indicator of whether the
machine failed due to one or more of five failure modes: Tool Wear
Failure (TWF), Heat Dissipation Failure (HDF), Power Failure (PWF),
Overstrain Failure (OSF), and Random Failures (RNF).

Initial data exploration confirmed the dataset has no missing values
and no duplicate records, making it largely analysis-ready. However,
a critical challenge was identified early: the dataset is severely
imbalanced, with only approximately 1.6% of records representing an
actual machine failure. This imbalance means that a naive model
predicting "no failure" for every record would achieve high accuracy
while being practically useless, as it would fail to catch any real
failures. Addressing this imbalance is therefore central to the
modeling strategy.

The project follows a structured pipeline: exploratory data analysis
to understand feature distributions and relationships with the
target variable; data wrangling and feature engineering to create
meaningful derived features (such as power, calculated from torque
and rotational speed); encoding of the categorical product type;
feature scaling; and handling of class imbalance using techniques
such as SMOTE (Synthetic Minority Oversampling Technique). Three
classification algorithms were experimented with - Logistic
Regression as a baseline, Random Forest, and XGBoost - to compare
performance across linear and tree-based/ensemble approaches.

Given the class imbalance, model evaluation prioritizes Recall and
F1-score over raw accuracy, since the business cost of missing an
actual failure (false negative) is far higher than the cost of a
false alarm (false positive). Hyperparameter tuning via GridSearchCV
was performed to optimize the best-performing model further. Feature
importance analysis was also conducted to identify which sensor
readings most strongly drive failure predictions, providing
actionable insight for TATA Steel's maintenance teams on where to
focus monitoring efforts.

The final deliverable is a trained classification model capable of
flagging machines at risk of failure before it occurs, along with
clear documentation of the reasoning behind each modeling decision.
This solution can help TATA Steel reduce unplanned downtime, lower
maintenance costs, and improve overall production reliability and
safety.

# **GitHub Link -**

https://github.com/rajputaman123/tata-steel-machine-failure-prediction.git

# **Problem Statement**


**TATA Steel operates a large number of heavy machines in its steel
production process. Unplanned machine failures lead to production
downtime, increased maintenance costs, and potential safety hazards.
The objective of this project is to build a machine learning
classification model that predicts whether a machine is likely to
fail based on real-time operational sensor readings such as air
temperature, process temperature, rotational speed, torque, and
tool wear. This enables proactive maintenance scheduling instead of
reactive repairs after a failure occurs.**

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report)
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

### Dataset Loading

In [ ]:
# Load Dataset
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

### Dataset First View

In [ ]:
# Dataset First Look
train_df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print("Rows:", train_df.shape[0])
print("Columns:", train_df.shape[1])

### Dataset Information

In [ ]:
# Dataset Info
train_df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Duplicate rows:", train_df.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
train_df.isnull().sum()

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10,4))
sns.heatmap(train_df.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Values Heatmap")
plt.show()

### What did you know about your dataset?



```
The dataset contains 136,429 rows and 14 columns with no missing
values and no duplicate rows, making it clean and ready for
analysis. It includes numerical sensor readings (air temperature,
process temperature, rotational speed, torque, tool wear), a
categorical product quality type (Low/Medium/High), and a binary
target variable "Machine failure" along with 5 individual failure
mode indicators (TWF, HDF, PWF, OSF, RNF). The dataset is severely
imbalanced, with only about 1.6% of records representing an actual
machine failure.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
train_df.columns

In [ ]:
# Dataset Describe
train_df.describe()

### Variables Description

- UDI/id: Unique row identifier (not used for modeling)
- Product ID: Unique product code (not used for modeling)
- Type: Product quality category (Low/Medium/High)
- Air temperature [K]: Ambient air temperature in Kelvin
- Process temperature [K]: Internal process temperature in Kelvin
- Rotational speed [rpm]: Machine rotational speed
- Torque [Nm]: Force applied by the machine tool
- Tool wear [min]: Cumulative usage time of the tool
- Machine failure: Target variable (1 = failure, 0 = no failure)
- TWF, HDF, PWF, OSF, RNF: Individual failure type indicators

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for col in train_df.columns:
    print(f"{col}: {train_df[col].nunique()} unique values")

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# Drop ID columns not useful for modeling
train_df = train_df.drop(columns=['id', 'Product ID'], errors='ignore')

# Clean column names for easier coding
train_df.columns = train_df.columns.str.replace(' ', '_').str.replace('[','').str.replace(']','')

train_df.head()